In [ ]:
!pip install pyspark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import os
import datetime

In [15]:
# 기존 Spark 세션이 남아 있다면 종료
spark = SparkSession.getActiveSession()
if spark:
    spark.stop()

In [3]:
# Spark 세션 생성 .master("spark://spark-master:7077")로 추후 수정 예정
spark = SparkSession.builder \
    .appName("W5M1_Analysis") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "LEGACY") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "80") \
    .config("spark.sql.files.maxPartitionBytes", "512m") \
    .getOrCreate()

sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/03 15:56:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
data_dir = "./data/NYC-TLC"
parquet_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".parquet")]

# 모든 Parquet 파일을 하나의 DataFrame으로 로드
df = spark.read.parquet(*parquet_files)

df = df.drop("airport_fee")

df = df.withColumn("tpep_pickup_datetime", to_timestamp(col("tpep_pickup_datetime")))
df = df.withColumn("tpep_dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime")))

df.printSchema()

25/02/03 15:56:59 WARN SQLConf: The SQL config 'spark.sql.legacy.parquet.datetimeRebaseModeInRead' has been deprecated in Spark v3.2 and may be removed in the future. Use 'spark.sql.parquet.datetimeRebaseModeInRead' instead.
25/02/03 15:56:59 WARN SQLConf: The SQL config 'spark.sql.legacy.parquet.datetimeRebaseModeInRead' has been deprecated in Spark v3.2 and may be removed in the future. Use 'spark.sql.parquet.datetimeRebaseModeInRead' instead.
25/02/03 15:56:59 WARN SQLConf: The SQL config 'spark.sql.legacy.parquet.datetimeRebaseModeInRead' has been deprecated in Spark v3.2 and may be removed in the future. Use 'spark.sql.parquet.datetimeRebaseModeInRead' instead.


root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [5]:
# DataFrame을 RDD로 변환
raw_rdd = df.rdd

In [6]:
raw_rdd.take(5)

25/02/03 15:57:02 WARN SQLConf: The SQL config 'spark.sql.legacy.parquet.datetimeRebaseModeInRead' has been deprecated in Spark v3.2 and may be removed in the future. Use 'spark.sql.parquet.datetimeRebaseModeInRead' instead.


[Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2020, 1, 1, 0, 28, 15), tpep_dropoff_datetime=datetime.datetime(2020, 1, 1, 0, 33, 3), passenger_count=1.0, trip_distance=1.2, RatecodeID=1.0, store_and_fwd_flag='N', PULocationID=238, DOLocationID=239, payment_type=1, fare_amount=6.0, extra=3.0, mta_tax=0.5, tip_amount=1.47, tolls_amount=0.0, improvement_surcharge=0.3, total_amount=11.27, congestion_surcharge=2.5),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2020, 1, 1, 0, 35, 39), tpep_dropoff_datetime=datetime.datetime(2020, 1, 1, 0, 43, 4), passenger_count=1.0, trip_distance=1.2, RatecodeID=1.0, store_and_fwd_flag='N', PULocationID=239, DOLocationID=238, payment_type=1, fare_amount=7.0, extra=3.0, mta_tax=0.5, tip_amount=1.5, tolls_amount=0.0, improvement_surcharge=0.3, total_amount=12.3, congestion_surcharge=2.5),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2020, 1, 1, 0, 47, 41), tpep_dropoff_datetime=datetime.datetime(2020, 1, 1, 0, 53, 52), passenge

In [7]:
# 컬럼명 출력 (인덱스와 함께 표시) (데이터 정제 전 컬럼 인덱스 확인 차)
columns = df.columns
print("📌 컬럼 인덱스 매핑:")
for i, col in enumerate(columns):
    print(f"{i}: {col}")

📌 컬럼 인덱스 매핑:
0: VendorID
1: tpep_pickup_datetime
2: tpep_dropoff_datetime
3: passenger_count
4: trip_distance
5: RatecodeID
6: store_and_fwd_flag
7: PULocationID
8: DOLocationID
9: payment_type
10: fare_amount
11: extra
12: mta_tax
13: tip_amount
14: tolls_amount
15: improvement_surcharge
16: total_amount
17: congestion_surcharge


In [8]:
rdd_no_missing = raw_rdd.filter(lambda row: all(x is not None for x in row))

In [9]:
# # 날짜 컬럼을 timestamp 변환 (tpep_pickup_datetime: 1번, tpep_dropoff_datetime: 2번)
# def convert_to_timestamp(row):
#     try:
#         pickup_dt = datetime.strptime(row[1], "%Y-%m-%d %H:%M:%S")
#         dropoff_dt = datetime.strptime(row[2], "%Y-%m-%d %H:%M:%S")
#         return (row[0], pickup_dt, dropoff_dt) + row[3:]  # 변환된 값 적용
#     except:
#         return None
    
# rdd_with_timestamp = rdd_no_missing.map(convert_to_timestamp).filter(lambda x: x is not None)

# rdd_with_timestamp.take(5)

In [10]:
# dropoff 시간이 pickup 시간보다 빠른 경우 제거
rdd_no_negative_time = rdd_no_missing.filter(lambda row: row[2] >= row[1])

In [11]:
# numeric_cols에서 음수 값이 있는 행 제거
numeric_indices = [3, 4, 5, 10, 11, 12, 13, 14, 15, 16, 17]  # 숫자형 컬럼 인덱스
rdd_no_negative_values = rdd_no_negative_time.filter(lambda row: all(row[i] >= 0 for i in numeric_indices))

In [ ]:
# fare_amount가 0인 경우 제거 (fare_amount 인덱스: 10번)
rdd_no_negative_values = rdd_no_negative_values.filter(lambda row: row[10] > 0)

In [ ]:
rdd_final = rdd_no_negative_values.filter(lambda row: row[1].year in [2020,2021])

In [13]:
# 캐싱 적용 (이전 Transformation 반복 방지)
rdd_final = rdd_final.cache()

In [14]:
# 연산 수행 및 샘플 데이터 확인
rdd_final.take(5)

25/02/03 15:57:12 WARN SQLConf: The SQL config 'spark.sql.legacy.parquet.datetimeRebaseModeInRead' has been deprecated in Spark v3.2 and may be removed in the future. Use 'spark.sql.parquet.datetimeRebaseModeInRead' instead.
ERROR:root:KeyboardInterrupt while sending command.                 (0 + 1) / 1]
Traceback (most recent call last):
  File "/opt/homebrew/anaconda3/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/anaconda3/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/anaconda3/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [18]:
# 1️⃣ 전체 운행 횟수 (count)
total_trips = rdd_final.count()

25/02/03 15:51:24 WARN SQLConf: The SQL config 'spark.sql.legacy.parquet.datetimeRebaseModeInRead' has been deprecated in Spark v3.2 and may be removed in the future. Use 'spark.sql.parquet.datetimeRebaseModeInRead' instead.
25/02/03 15:51:24 WARN SQLConf: The SQL config 'spark.sql.legacy.parquet.datetimeRebaseModeInRead' has been deprecated in Spark v3.2 and may be removed in the future. Use 'spark.sql.parquet.datetimeRebaseModeInRead' instead.
25/02/03 15:51:24 WARN SQLConf: The SQL config 'spark.sql.legacy.parquet.datetimeRebaseModeInRead' has been deprecated in Spark v3.2 and may be removed in the future. Use 'spark.sql.parquet.datetimeRebaseModeInRead' instead.
25/02/03 15:51:24 WARN SQLConf: The SQL config 'spark.sql.legacy.parquet.datetimeRebaseModeInRead' has been deprecated in Spark v3.2 and may be removed in the future. Use 'spark.sql.parquet.datetimeRebaseModeInRead' instead.
25/02/03 15:51:24 WARN SQLConf: The SQL config 'spark.sql.legacy.parquet.datetimeRebaseModeInRead' h

KeyboardInterrupt: 

In [ ]:
# 2️⃣ 총 수익 (운임의 합) - reduce 사용
total_revenue = rdd_final.map(lambda row: row[16]).reduce(lambda a, b: a + b)

In [ ]:
# 3️⃣ 평균 운행 거리 - aggregateByKey 사용 (합계, 개수 -> 평균)
distance_stats = rdd_final.map(lambda row: (1, row[4])) \
                          .aggregateByKey((0, 0),
                                          lambda acc, value: (acc[0] + value, acc[1] + 1),  # (거리 합, 개수)
                                          lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1]))  # 파티션 병합

total_distance, count_distance = distance_stats.collect()[0][1]
avg_distance = total_distance / count_distance if count_distance > 0 else 0

In [ ]:
# 3️⃣ 평균 운행 거리 - reduceByKey 사용 (합계, 개수 -> 평균)
distance_stats = rdd_final.map(lambda row: (1, (row[4], 1))) \
                          .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
total_distance, count_distance = distance_stats.collect()[0][1]
avg_distance = total_distance / count_distance

In [ ]:
# 4️⃣ 일별 운행 횟수 - reduceByKey 사용
daily_trips = rdd_final.map(lambda row: (row[1].strftime("%Y-%m-%d"), 1)) \
                       .reduceByKey(lambda a, b: a + b)

In [ ]:
# 5️⃣ 일별 총 수익 - reduceByKey 사용
daily_revenue = rdd_final.map(lambda row: (row[1].strftime("%Y-%m-%d"), row[16])) \
                         .reduceByKey(lambda a, b: a + b)

In [ ]:
# 6️⃣ 운행 횟수 & 총 수익 RDD Join
daily_summary = daily_trips.join(daily_revenue).collect()

In [ ]:
# ✅ 결과 출력
print(f"📌 전체 운행 횟수: {total_trips}")
print(f"📌 총 수익: {total_revenue:.2f} USD")
print(f"📌 평균 운행 거리: {avg_distance:.2f} miles")

print("\n📌 일별 운행 요약:")
for date, (trips, revenue) in sorted(daily_summary):
    print(f"{date}: 운행 {trips}건, 총 수익 {revenue:.2f} USD")

In [ ]:
!pip install boto3

In [ ]:
import pandas as pd
import os
import boto3

# ✅ 환경 변수에서 AWS Credentials & S3 버킷 이름 가져오기
AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = "ap-northeast-2"
S3_BUCKET = "softeer-w5-bucket"

# ✅ S3 저장 경로 설정
s3_summary_path = f"s3://{S3_BUCKET}/nyc_taxi_summary.csv"
s3_daily_summary_path = f"s3://{S3_BUCKET}/nyc_taxi_daily_summary.csv"

# ✅ 로컬 저장 경로 설정
local_summary_path = "data/nyc_taxi_summary.csv"
local_daily_summary_path = "data/nyc_taxi_daily_summary.csv"

# ✅ 분석 결과 DataFrame 변환
df_summary = pd.DataFrame({
    "total_trips": [total_trips],
    "total_revenue": [total_revenue],
    "avg_distance": [avg_distance]
})

# ✅ 데이터 변환 (튜플 리스트 → Pandas DataFrame)
df_daily_summary = pd.DataFrame(
    [(date, trips, revenue) for date, (trips, revenue) in daily_summary],
    columns=["date", "total_trips", "total_revenue"]
)

df_summary.to_csv(local_summary_path, index=False)
df_daily_summary.to_csv(local_daily_summary_path, index=False)


print(f"✅ CSV 파일이 로컬에 저장됨: {local_summary_path}")
print(f"✅ CSV 파일이 로컬에 저장됨: {local_daily_summary_path}")

# ✅ boto3 클라이언트 생성
s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

# ✅ S3에 업로드할 파일 경로
s3_summary_key = "nyc_taxi_summary.csv"
s3_daily_summary_key = "nyc_taxi_daily_summary.csv"

# ✅ S3에 업로드
s3_client.upload_file(local_summary_path, S3_BUCKET, s3_summary_key)
s3_client.upload_file(local_daily_summary_path, S3_BUCKET, s3_daily_summary_key)

print(f"✅ CSV 파일이 S3에 업로드됨: s3://{S3_BUCKET}/{s3_summary_key}")
print(f"✅ CSV 파일이 S3에 업로드됨: s3://{S3_BUCKET}/{s3_daily_summary_key}")